In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import pandas as pd
import cv2
import os
import numpy as np
from pathlib import Path

import sys
sys.path.append('..')

from Pipeline import Pipeline

**Helper functions**

In [ ]:
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union for two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-5)
    return iou

def yolo_to_xyxy(x_center, y_center, w, h, img_w, img_h):
    """Convert YOLO format to [x1, y1, x2, y2]"""
    return [
        int((x_center - w/2) * img_w),
        int((y_center - h/2) * img_h),
        int((x_center + w/2) * img_w),
        int((y_center + h/2) * img_h)
    ]

**Initialization**

In [ ]:
# Initialize Pipeline
print("Loading Pipeline...")
pipeline = Pipeline()

# Load Test CSV
csv_path = "Test_dataset.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df['image_filename'].unique())} unique test images.")

**Testing the pipeline**

In [ ]:
# Metrics dictionaries
metrics = {
    "border_correct": 0,
    "border_total": 0,
    
    "fp_border_but_zero_notches": 0, 
    "fp_border_with_notches": 0,
    
    "notch_true_positives": 0,
    "notch_false_positives": 0, 
    "notch_false_negatives": 0,
    
    "shape_correct": 0,
    "shape_total": 0,

    "slope_total": 0,
    "direction_correct": 0,

    "human_review_total": 0,
    "human_review_right": 0,
    "human_review_wrong": 0,
    "wrongly_not_flagged_for_review": 0,

    "code_correct": 0,
    "code_total": 0,

    'border_code_total': 0,
    'border_code_correct': 0,

    'code_wrong_without_review': 0
}

missed_notches_list = []    
false_positives_list = []  
wrong_shapes_list = []
wrong_codes_list = []

IOU_THRESHOLD = 0.15

# Group by image so we process each image only once
for img_file, group in df.groupby('image_filename'):
    # Read Image
    subfolder = img_file[0].upper()
    img_path = os.path.join(path_to_dataset, subfolder, img_file)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read {img_path}. Skipping.")
        continue

    img_h, img_w = img.shape[:2]
    # Get Ground Truth (GT)
    gt_has_border = str(group.iloc[0]['has_border']).lower() in ['true', '1']
    gt_code = str(group.iloc[0]['notch_code']).strip().lower()
    
    # Run Pipeline
    results = pipeline.process_image(img)
    
    # --- 1. Evaluate Border Detection ---
    pred_has_border = results['border_present']
    if pred_has_border == gt_has_border:
        metrics["border_correct"] += 1
    elif pred_has_border and not gt_has_border:
        if len(results['accepted_notches']) == 0:
            metrics["fp_border_but_zero_notches"] += 1
        else:
            metrics["fp_border_with_notches"] += 1
            
    metrics["border_total"] += 1

    # --- 2. Evaluate complete notch code ---
    if not pred_has_border or not results['accepted_notches']:
        pred_code = "none"
    else:
        pred_code, used_notches = pipeline.generate_notch_code(results['accepted_notches'], img.shape)

    any_notch_flagged = any(n.get('needs_human_review', False) for n in used_notches)

    metrics["code_total"] += 1
    if pred_code == gt_code:
        metrics["code_correct"] += 1
    else:
        if not any_notch_flagged:
                    metrics["code_wrong_without_review"] += 1

        wrong_codes_list.append({
            "image_filename": img_file,
            "gt_code": gt_code,
            "pred_code": pred_code,
            "human_review": any_notch_flagged
        })

    if gt_has_border:
        metrics["border_code_total"] += 1
        if pred_code == gt_code:
            metrics["border_code_correct"] += 1

    # --- 3 & 4. Evaluate Notch Detection & Shape ---
    # Extract GT notches (ignore rows labeled 'none' or noise)
    gt_notches = group[group['super_category'].str.lower() == 'notch']
    predicted_notches = results['accepted_notches'].copy()
    
    matched_preds = set()
    
    # Check for True Positives and False Negatives
    for _, gt_row in gt_notches.iterrows():
        gt_box = yolo_to_xyxy(gt_row['x_center'], gt_row['y_center'], gt_row['width'], gt_row['height'], img_w, img_h)

        raw_gt_shape = str(gt_row['fine_category']).lower()
        if raw_gt_shape in ['sloped_left', 'sloped_right']:
            gt_shape = 'sloped'
            gt_direction = raw_gt_shape
        else:
            gt_shape = raw_gt_shape
            gt_direction = None
        
        best_iou = 0
        best_pred_idx = -1
        
        # Find the best matching prediction
        for idx, pred in enumerate(predicted_notches):
            if idx in matched_preds:
                continue
            iou = calculate_iou(pred['coords'], gt_box)
            if iou > best_iou:
                best_iou = iou
                best_pred_idx = idx
        
        if best_iou >= IOU_THRESHOLD:
            # True Positive Notch
            metrics["notch_true_positives"] += 1
            matched_preds.add(best_pred_idx)
            
            # Check Grouped Shape Accuracy
            pred_data = predicted_notches[best_pred_idx]
            pred_shape = str(pred_data['shape']).lower()            
            needs_review = pred_data.get('needs_human_review', False)

            is_fully_correct = False

            if pred_shape == gt_shape:
                metrics["shape_correct"] += 1
                is_fully_correct = True

                if gt_shape == 'sloped':
                    metrics["slope_total"] += 1
                    pred_direction = str(predicted_notches[best_pred_idx].get('slope_direction', '')).lower()
                    if pred_direction == gt_direction:
                        metrics["direction_correct"] += 1
                    else:
                        wrong_shapes_list.append({
                            "image_filename": img_file,
                            "gt_box": gt_box,
                            "pred_box": predicted_notches[best_pred_idx]['coords'],
                            "gt_shape": gt_direction,  
                            "pred_shape": pred_direction,
                            "error_type": "direction_mismatch"
                        })
                    

            else:
                wrong_shapes_list.append({
                    "image_filename": img_file,
                    "gt_box": gt_box,
                    "pred_box": predicted_notches[best_pred_idx]['coords'],
                    "gt_shape": gt_shape,
                    "pred_shape": pred_shape,
                    "error_type": "shape_mismatch"
                })
            metrics["shape_total"] += 1

            if needs_review:
                metrics["human_review_total"] += 1
                if is_fully_correct:
                    metrics["human_review_right"] += 1
                else:
                    metrics["human_review_wrong"] += 1
            else:
                if not is_fully_correct:
                    metrics['wrongly_not_flagged_for_review'] += 1
                                       
        else:
            # GT notch missed by pipeline (False Negative)
            metrics["notch_false_negatives"] += 1
            missed_notches_list.append({
                "image_filename": img_file,
                "gt_box": gt_box,
                "gt_shape": raw_gt_shape
            })
                        
    # Any predicted notches that didn't match a GT notch are False Positives (Noise classified as Notch)
    for idx, pred in enumerate(predicted_notches):
        if idx not in matched_preds:
            # TRACK: False Positive
            false_positives_list.append({
                "image_filename": img_file,
                "pred_box": pred['coords'],
                "pred_shape": pred.get('shape', 'notch'),
                "conf": pred.get('shape_confidence', 0)
            })

            if pred.get('needs_human_review', False):
                metrics["human_review_total"] += 1
                metrics["human_review_wrong"] += 1

    metrics["notch_false_positives"] += len(predicted_notches) - len(matched_preds)

print("Evaluation Complete!")
print(f"Captured {len(missed_notches_list)} missed notches, {len(false_positives_list)} false positives, and {len(wrong_shapes_list)} shape/direction, and {len(wrong_codes_list)} full-code sequence errors.")

**Results**

In [ ]:
# Calculate Summary Statistics
b_acc = (metrics["border_correct"] / metrics["border_total"]) * 100 if metrics["border_total"] > 0 else 0

tp = metrics["notch_true_positives"]
fp = metrics["notch_false_positives"]
fn = metrics["notch_false_negatives"]

precision = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0
recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

# Loose: Base shape matches (sloped == sloped)
shape_correct_loose = metrics["shape_correct"]
s_acc_loose = (shape_correct_loose / metrics["shape_total"]) * 100 if metrics["shape_total"] > 0 else 0

# Direction specifics
slope_total = metrics["slope_total"]
dir_correct = metrics["direction_correct"]
dir_mismatch = slope_total - dir_correct
dir_acc = (dir_correct / slope_total) * 100 if slope_total > 0 else 0

# Strict: Base shape matches AND direction matches
shape_correct_strict = shape_correct_loose - dir_mismatch
s_acc_strict = (shape_correct_strict / metrics["shape_total"]) * 100 if metrics["shape_total"] > 0 else 0

print("="*50)
print(" 📊 PIPELINE EVALUATION RESULTS")
print("="*50)
print(f"1. Border Detection")
print(f"   Overall Accuracy          : {b_acc:.1f}% ({metrics['border_correct']}/{metrics['border_total']})")
print(f"   -> FP Borders (0 notches predicted) : {metrics['fp_border_but_zero_notches']}")
print(f"   -> FP Borders (>0 notches predicted): {metrics['fp_border_with_notches']}")
print("-" * 50)
print(f"2. Notch Detection (SVM 1) ")
print(f"   Precision                 : {precision:.1f}%")
print(f"   Recall                    : {recall:.1f}%")
print(f"   True Positives (Found GT) : {tp}")
print(f"   False Positives (Noise)   : {fp}")
print(f"   False Negatives (Missed)  : {fn}")
print("-" * 50)
print(f"3. Shape Classification (SVM 2 - Grouped)")
print(f"   Loose Accuracy (Ignore Direction): {s_acc_loose:.1f}% ({shape_correct_loose}/{metrics['shape_total']})")
print(f"   Strict Accuracy (With Direction) : {s_acc_strict:.1f}% ({shape_correct_strict}/{metrics['shape_total']})")
print(f"   -> Cost of Direction Mismatches : -{s_acc_loose - s_acc_strict:.1f}% ({dir_mismatch} errors)")
print("-" * 50)
print(f"4. Slope Direction Classification (Left / Right)")
print(f"   Accuracy                  : {dir_acc:.1f}% ({dir_correct}/{slope_total})")
print("-"*50)
print(f"5. Human Review (Confidence < 0.65)")
print(f"   Total Flags Raised        : {metrics['human_review_total']}")
if metrics['human_review_total'] > 0:
    hr_right = metrics['human_review_right']
    hr_wrong = metrics['human_review_wrong']
    hr_missed = metrics['wrongly_not_flagged_for_review']
    print(f"   -> Model was actually RIGHT : {hr_right} ({(hr_right/metrics['human_review_total'])*100:.1f}%)")
    print(f"   -> Model was actually WRONG : {hr_wrong} ({(hr_wrong/metrics['human_review_total'])*100:.1f}%)")
print(f"   -> Model failed to raise {hr_missed} flags")
print("-"*50)
if metrics["code_total"] > 0:
    overall_code_acc = (metrics["code_correct"] / metrics["code_total"]) * 100
    print(f"Overall Notch Code Accuracy (All Images): {overall_code_acc:.2f}% ({metrics['code_correct']}/{metrics['code_total']})")
if metrics["border_code_total"] > 0:
    border_code_acc = (metrics["border_code_correct"] / metrics["border_code_total"]) * 100
    print(f"Notch Code Accuracy (Images With Border Only): {border_code_acc:.2f}% ({metrics['border_code_correct']}/{metrics['border_code_total']})")
print(f"-> Confidently Wrong Codes (Wrong code AND 0 notches flagged for review): {metrics['code_wrong_without_review']}")
print("="*50)

**Visualisation of mistakes**

**See what notches it missed and what false positives it produced**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import os

%matplotlib qt 

def interactive_error_viewer(missed_list, fp_list, path_to_dataset):
    # Group errors by image
    errors_by_image = {}
    
    for item in missed_list:
        img = item['image_filename']
        if img not in errors_by_image: errors_by_image[img] = {'missed': [], 'fps': []}
        errors_by_image[img]['missed'].append(item)
        
    for item in fp_list:
        img = item['image_filename']
        if img not in errors_by_image: errors_by_image[img] = {'missed': [], 'fps': []}
        errors_by_image[img]['fps'].append(item)

    image_files = list(errors_by_image.keys())
    
    if not image_files:
        print("No detection errors to visualize! Perfect score.")
        return

    # State variables
    current_idx = 0
    total_images = len(image_files)

    # Setup the plot figure
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.canvas.manager.set_window_title('Notch Detection Error Viewer')

    def draw_image(idx):
        """Clears the axis and draws the requested image and boxes."""
        ax.clear()
        img_file = image_files[idx]
        data = errors_by_image[img_file]
        
        subfolder = img_file[0].upper()
        full_img_path = os.path.join(path_to_dataset, subfolder, img_file)
        img = cv2.imread(full_img_path)
        
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        else:
            ax.text(0.5, 0.5, "Image file not found on disk", ha='center', fontsize=14)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)

        # Set dynamic title
        ax.set_title(f"[{idx + 1} / {total_images}] File: {img_file}\n"
                     f"Missed (Orange): {len(data['missed'])} | FPs (Red): {len(data['fps'])}\n"
                     "(Use Left/Right Arrows. Press 'Esc' to close)", 
                     fontsize=12, color='darkred', weight='bold')
        
        # Draw Missed (Orange)
        for m in data['missed']:
            x1, y1, x2, y2 = m['gt_box']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='orange', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 8, f"MISSED: {m['gt_shape']}", color='red', fontsize=5, weight='bold')
            
        # Draw False Positives (Red Dashed)
        for fp in data['fps']:
            x1, y1, x2, y2 = fp['pred_box']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='red', facecolor='none', linestyle='--')
            ax.add_patch(rect)
            ax.text(x1, y2 + 18, f"FP: {fp['pred_shape']} ({fp['conf']:.2f})", color='blue', fontsize=5, weight='bold')
        
        ax.axis('off')
        fig.canvas.draw_idle() # Update the canvas

    def on_key_press(event):
        """Handles keyboard navigation."""
        nonlocal current_idx
        
        if event.key == 'right':
            current_idx = (current_idx + 1) % total_images
            draw_image(current_idx)
        elif event.key == 'left':
            current_idx = (current_idx - 1) % total_images
            draw_image(current_idx)
        elif event.key == 'escape':
            plt.close(fig)

    # Connect the keyboard event to the figure
    fig.canvas.mpl_connect('key_press_event', on_key_press)

    # Draw the first image
    draw_image(current_idx)
    plt.tight_layout()
    plt.show()

# Run the interactive viewer
interactive_error_viewer(missed_notches_list, false_positives_list, path_to_dataset)

**See confusion matrix**

In [ ]:
import seaborn as sns

%matplotlib inline

def plot_error_confusion_matrix(wrong_shapes_list):
    if not wrong_shapes_list:
        print("No shape misclassifications to plot! Perfect accuracy.")
        return
        
    # Convert our error list into a Pandas DataFrame
    df_errors = pd.DataFrame(wrong_shapes_list)
    
    # Create a cross-tabulation (frequency table) of Ground Truth vs Predictions
    # This automatically builds the matrix layout
    confusion_df = pd.crosstab(
        df_errors['gt_shape'], 
        df_errors['pred_shape'], 
        dropna=False
    )
    
    # Plot using Seaborn Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_df, annot=True, fmt='d', cmap='Reds', cbar=True,
                linewidths=1, linecolor='black')
    
    plt.title('Misclassification Heatmap (Only Errors Shown)', fontsize=16, pad=15)
    plt.xlabel('What the Model Predicted', fontsize=12, labelpad=10)
    plt.ylabel('Ground Truth', fontsize=12, labelpad=10)
    
    # Move X-axis labels to top for easier reading
    plt.gca().xaxis.tick_top()
    plt.gca().xaxis.set_label_position('top')
    
    plt.tight_layout()
    plt.show()

# Run the confusion matrix
plot_error_confusion_matrix(wrong_shapes_list)

**See what notch codes were wrong**

In [ ]:
%matplotlib qt 

def interactive_code_error_viewer(wrong_codes_list, pipeline, path_to_dataset):
    """
    Interactive viewer for notch code sequence errors.
    Uses Left/Right arrow keys to navigate through images with code mismatches, 
    and 'Esc' to close.
    """
    if not wrong_codes_list:
        print("No notch code sequence errors to visualize! Perfect score.")
        return

    image_files = wrong_codes_list
    current_idx = 0
    total_images = len(image_files)

    # Setup the plot figure
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.canvas.manager.set_window_title('Notch Code Sequence Error Viewer')

    def draw_image(idx):
        """Clears the axis, runs the pipeline for the image, and displays boxes + code mismatch info."""
        ax.clear()
        error_item = image_files[idx]
        img_file = error_item['image_filename']
        gt_code = error_item['gt_code']
        pred_code = error_item['pred_code']
        
        subfolder = img_file[0].upper()
        full_img_path = os.path.join(path_to_dataset, subfolder, img_file)
        img = cv2.imread(full_img_path)
        
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            
            # Run the pipeline to get current predictions for visualization
            results = pipeline.process_image(img)
            accepted_notches = results.get('accepted_notches', [])
            
            # Draw predicted bounding boxes and shapes
            for pred in accepted_notches:
                x1, y1, x2, y2 = pred['coords']
                shape = pred.get('shape', 'unknown')
                conf = pred.get('shape_confidence', 0.0)
                
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='cyan', facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 8, f"{shape} ({conf:.2f})", color='yellow', fontsize=5, weight='bold')
        else:
            ax.text(0.5, 0.5, "Image file not found on disk", ha='center', fontsize=14)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)

        # Set dynamic title highlighting the mismatch
        ax.set_title(f"[{idx + 1} / {total_images}] File: {img_file}\n"
                     f"GT Code: '{gt_code}'  |  Predicted Code: '{pred_code}'\n"
                     "(Use Left/Right Arrows. Press 'Esc' to close)", 
                     fontsize=12, color='darkred', weight='bold')
        
        ax.axis('off')
        fig.canvas.draw_idle() # Update the canvas

    def on_key_press(event):
        """Handles keyboard navigation."""
        nonlocal current_idx
        
        if event.key == 'right':
            current_idx = (current_idx + 1) % total_images
            draw_image(current_idx)
        elif event.key == 'left':
            current_idx = (current_idx - 1) % total_images
            draw_image(current_idx)
        elif event.key == 'escape':
            plt.close(fig)

    # Connect the keyboard event to the figure
    fig.canvas.mpl_connect('key_press_event', on_key_press)

    # Draw the first image
    draw_image(current_idx)
    plt.tight_layout()
    plt.show()

# Run the interactive code error viewer
#FIXME: Comment out next line if you want to see all the wrong notch codes
wrong_codes_no_review = [item for item in wrong_codes_list if not item.get('human_review', False)]
interactive_code_error_viewer(wrong_codes_no_review, pipeline, path_to_dataset)